In [ ]:
#1. 导入需要使用的 Python 库

from pathlib import Path
import time

import duckdb
from IPython.display import display

In [ ]:
RAW_FILE = Path("../data/raw/UserBehavior.csv")

# DuckDB 数据库与原始 CSV 分开保存。
DB_FILE = Path("../data/database/taobao.duckdb")

# 清洗、聚合后的 CSV 保存目录。
PROCESSED_DIR = Path("../data/processed")

# DuckDB 执行大数据操作时的临时文件目录。
TEMP_DIR = Path("../data/temp")

# Python 绘图输出目录。
FIGURE_DIR = Path("../outputs/figures")

# 分析报告保存目录。
DOCS_DIR = Path("../docs")

In [ ]:
# 这里传入的是 .duckdb 文件，不是原始 CSV。
con = duckdb.connect(str(DB_FILE))


In [ ]:
raw_path_sql = RAW_FILE.resolve().as_posix().replace("'", "''")

# f"""...""" 表示允许把 Python 变量放进多行字符串。
query_raw = f"""
CREATE OR REPLACE VIEW raw_behavior AS

SELECT *
FROM read_csv(
    '{raw_path_sql}',
    header = false,
    delim = ',',
    columns = {{
        'user_id': 'VARCHAR',
        'item_id': 'VARCHAR',
        'category_id': 'VARCHAR',
        'behavior_type': 'VARCHAR',
        'timestamp': 'VARCHAR'
    }}
)
"""

con.execute(query_raw)

# 只展示前 5 行。
display(
    con.execute("""
        SELECT *
        FROM raw_behavior
        LIMIT 5
    """).df()
)

In [ ]:
#清理前后空格、统一行为类型大小写、把 id 和时间戳安全转成数字，脏数据自动变成 NULL 而不崩溃
query_typed = """
CREATE OR REPLACE VIEW typed_behavior AS

SELECT
    TRY_CAST(TRIM(user_id) AS BIGINT) AS user_id,
    TRY_CAST(TRIM(item_id) AS BIGINT) AS item_id,
    TRY_CAST(TRIM(category_id) AS BIGINT) AS category_id,

    LOWER(TRIM(behavior_type)) AS behavior_type,

    TRY_CAST(TRIM(timestamp) AS BIGINT) AS ts

FROM raw_behavior
"""

con.execute(query_typed)

display(
    con.execute("""
        SELECT *
        FROM typed_behavior
        LIMIT 5
    """).df()
)

In [ ]:
#统计脏数据，空值/错误/不在时间范围内
query_quality = """
SELECT
    COUNT(*) AS raw_rows,

    COUNT(*) FILTER (
        WHERE user_id IS NULL
           OR item_id IS NULL
           OR category_id IS NULL
           OR ts IS NULL
    ) AS missing_or_cast_failed_rows,

    COUNT(*) FILTER (
        WHERE behavior_type IS NULL
           OR behavior_type NOT IN ('pv', 'cart', 'fav', 'buy')
    ) AS invalid_behavior_rows,

    COUNT(*) FILTER (
        WHERE ts < 1511539200
           OR ts >= 1512316800
    ) AS out_of_window_rows

FROM typed_behavior
"""

quality = con.execute(query_quality).df()

display(quality)

In [ ]:
#丢掉脏数据
query_valid = """
CREATE OR REPLACE VIEW valid_behavior AS

SELECT *
FROM typed_behavior

WHERE user_id IS NOT NULL
  AND item_id IS NOT NULL
  AND category_id IS NOT NULL
  AND ts IS NOT NULL

  AND behavior_type IN ('pv', 'cart', 'fav', 'buy')

  AND ts >= 1511539200
  AND ts < 1512316800
"""

con.execute(query_valid)

display(
    con.execute("""
        SELECT *
        FROM valid_behavior
        LIMIT 5
    """).df()
)

In [ ]:
'''
display(
    con.execute("""
        SELECT
            ts,

            timezone(
                'Asia/Shanghai',
                to_timestamp(ts)
            ) AS beijing_time

        FROM sample_after
        LIMIT 5
    """).df()
)
'''

In [ ]:
display(con.execute("SHOW TABLES").df())


In [ ]:
display(
    con.execute("DESCRIBE user_behavior").df()
)

display(
    con.execute("""
        SELECT *
        FROM user_behavior
        LIMIT 5
    """).df()
)

In [ ]:
# 统计筛选后、去重前的记录数。
valid_rows = con.execute("""
    SELECT COUNT(*)
    FROM valid_behavior
""").fetchone()[0]

# 统计清洗后的记录数。
clean_rows = con.execute("""
    SELECT COUNT(*)
    FROM user_behavior
""").fetchone()[0]

# 原始记录数来自前面的数据质量检查。
raw_rows = int(quality.loc[0, "raw_rows"])

print(f"原始记录数：{raw_rows:,}")
print(f"有效记录数，去重前：{valid_rows:,}")
print(f"清洗后记录数：{clean_rows:,}")

print(f"因字段、行为或时间规则排除：{raw_rows - valid_rows:,}")
print(f"完全重复去重减少：{valid_rows - clean_rows:,}")

# 基本逻辑检查：清洗后不能比原始记录更多。
assert raw_rows >= valid_rows >= clean_rows

print("行数关系检查通过。")

In [ ]:
# 清理中间 VIEW,让 db 文件自包含(02 及以后只依赖 user_behavior)
for view in ("valid_behavior", "typed_behavior", "raw_behavior"):
    con.execute(f"DROP VIEW IF EXISTS {view}")

con.close()
print("已清理中间 VIEW,并关闭数据库连接。")